# Implement Mixture of Experts Layer

**Difficulty**: 🔴 Hard

**Companies**: Google, DeepMind, Mistral, Databricks, xAI

---

### Problem Statement

**Mixture of Experts (MoE)** is a technique for scaling model capacity without proportionally scaling compute. Instead of one large feed-forward network (FFN), MoE uses multiple smaller "expert" FFNs and a **gating/router network** that selects the **top-k** experts for each token. This means each token is only processed by a fraction of the total parameters.

A critical challenge in MoE is **load balancing**: without regularization, the router may collapse to always selecting the same expert(s), leaving others unused. An **auxiliary load balancing loss** encourages the router to distribute tokens evenly across experts.

Your task is to implement:
1. An `Expert` FFN module.
2. A `MoELayer` with top-k gating, expert dispatch, and load balancing loss.
3. A training loop that demonstrates the loss decreasing and experts receiving balanced load.

---

### Requirements

1. **`Expert(nn.Module)`** — A simple two-layer FFN: Linear → ReLU → Linear.
2. **`MoELayer(nn.Module)`** — Contains a list of experts, a router (linear layer), and implements top-k gating with weighted combination of expert outputs.
3. **Load balancing loss** — Auxiliary loss = `num_experts * mean(fraction_tokens_per_expert * mean_gate_prob_per_expert)`. This penalizes uneven distribution.
4. **`MoEConfig`** — Dataclass with `num_experts`, `top_k`, `d_model`, `d_ff`, `aux_loss_weight`.

---

### Constraints

- ✅ Each token should be routed to exactly `top_k` experts.
- ✅ Expert outputs should be weighted by the (softmax-normalized) gate values.
- ✅ All experts should receive some tokens after training (no expert collapse).
- ✅ Total MoE parameters should be significantly more than a dense FFN, but FLOPs per token should be comparable to a single expert.
- ❌ Do **not** use any external MoE libraries.

---

<details>
  <summary>💡 Hint</summary>

  - Router: `gate_logits = self.router(x)` of shape `(batch*seq, num_experts)`. Use `torch.topk` to get top-k expert indices and values.
  - Normalize gate values: apply softmax only over the selected top-k values, not all experts.
  - Dispatch: loop over experts, find which tokens selected each expert, process them, and accumulate weighted outputs.
  - Load balance loss: `f_i` = fraction of tokens routed to expert i, `p_i` = mean router probability for expert i. Loss = `num_experts * sum(f_i * p_i)`.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

In [ ]:
@dataclass
class MoEConfig:
    num_experts: int = 8
    top_k: int = 2
    d_model: int = 64
    d_ff: int = 128
    aux_loss_weight: float = 0.01

config = MoEConfig()
print(f"Config: {config}")

In [ ]:
class Expert(nn.Module):
    """A simple feed-forward expert network."""

    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        # TODO: Create a two-layer FFN: Linear(d_model, d_ff) -> ReLU -> Linear(d_ff, d_model)
        ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO: Pass x through the FFN
        ...


class MoELayer(nn.Module):
    """Mixture of Experts layer with top-k gating and load balancing."""

    def __init__(self, config: MoEConfig):
        super().__init__()
        self.config = config

        # TODO: Create a list of Expert modules (use nn.ModuleList)

        # TODO: Create the router: a Linear layer (d_model -> num_experts)

        ...

    def forward(self, x: torch.Tensor):
        """
        Forward pass with top-k expert routing.

        Args:
            x: (batch_size, seq_len, d_model)

        Returns:
            output: (batch_size, seq_len, d_model)
            aux_loss: scalar load balancing loss
        """
        batch_size, seq_len, d_model = x.shape
        # Flatten to (num_tokens, d_model) for routing
        x_flat = x.view(-1, d_model)
        num_tokens = x_flat.size(0)

        # TODO: Step 1 - Compute router logits: (num_tokens, num_experts)

        # TODO: Step 2 - Get top-k experts per token
        #       Use torch.topk on gate_logits to get top_k_values and top_k_indices

        # TODO: Step 3 - Normalize gate values (softmax over the top-k values only)

        # TODO: Step 4 - Dispatch tokens to experts and combine outputs
        #       Initialize output = zeros_like(x_flat)
        #       For each expert i:
        #         - Find which (token, slot) pairs selected expert i
        #         - Get those tokens, run through expert i
        #         - Multiply by gate weight, add to output

        # TODO: Step 5 - Compute auxiliary load balancing loss
        #       router_probs = softmax(gate_logits)  -- over ALL experts
        #       f_i = fraction of tokens routed to expert i (from top_k_indices)
        #       p_i = mean router probability for expert i
        #       aux_loss = num_experts * sum(f_i * p_i)

        # TODO: Reshape output back to (batch_size, seq_len, d_model)

        ...


# Quick test
torch.manual_seed(42)
moe = MoELayer(config)
x_test = torch.randn(4, 8, config.d_model)  # batch=4, seq=8
out, aux_loss = moe(x_test)
print(f"Input shape:  {x_test.shape}")
print(f"Output shape: {out.shape}")
print(f"Aux loss:     {aux_loss.item():.4f}")

In [ ]:
# Training and validation
torch.manual_seed(42)

# Synthetic data: simple regression task
batch_size = 16
seq_len = 8
x_data = torch.randn(batch_size, seq_len, config.d_model)
# Target: a nonlinear transformation (sum of sine waves)
y_data = torch.sin(x_data[:, :, :config.d_model]) * 0.5

moe = MoELayer(config)
optimizer = torch.optim.Adam(moe.parameters(), lr=1e-3)

losses = []
num_steps = 300

for step in range(num_steps):
    optimizer.zero_grad()
    output, aux_loss = moe(x_data)
    task_loss = F.mse_loss(output, y_data)
    total_loss = task_loss + config.aux_loss_weight * aux_loss
    total_loss.backward()
    optimizer.step()
    losses.append(total_loss.item())
    if step % 75 == 0:
        print(f"Step {step:3d} | Task loss: {task_loss.item():.4f} | Aux loss: {aux_loss.item():.4f} | Total: {total_loss.item():.4f}")

print(f"\nFinal total loss: {losses[-1]:.4f}")

# Validation 1: Loss should decrease
assert losses[-1] < losses[0], "Loss did not decrease!"
print("PASSED: Loss decreased.\n")

# Validation 2: Check load balance - all experts should receive tokens
print("=== Expert Load Balance ===")
with torch.no_grad():
    x_flat = x_data.view(-1, config.d_model)
    gate_logits = moe.router(x_flat)
    _, top_k_indices = torch.topk(gate_logits, config.top_k, dim=-1)
    expert_counts = torch.zeros(config.num_experts)
    for i in range(config.num_experts):
        expert_counts[i] = (top_k_indices == i).sum().item()
    total_assignments = top_k_indices.numel()
    for i in range(config.num_experts):
        pct = expert_counts[i] / total_assignments * 100
        print(f"  Expert {i}: {expert_counts[i]:5.0f} tokens ({pct:.1f}%)")
    assert (expert_counts > 0).all(), "Some experts received zero tokens (expert collapse)!"
    print("PASSED: All experts received tokens.\n")

# Validation 3: Parameter count comparison
print("=== Parameter Count Comparison ===")
moe_params = sum(p.numel() for p in moe.parameters())
dense_ffn = nn.Sequential(nn.Linear(config.d_model, config.d_ff), nn.ReLU(), nn.Linear(config.d_ff, config.d_model))
dense_params = sum(p.numel() for p in dense_ffn.parameters())
print(f"  MoE parameters:       {moe_params:,}")
print(f"  Dense FFN parameters: {dense_params:,}")
print(f"  Ratio: {moe_params / dense_params:.1f}x more parameters in MoE")
print(f"  But only top-{config.top_k}/{config.num_experts} experts active per token!")
print("PASSED\n")

print("All tests passed!")